# Maximum-profile coverage · 1.4.3

Verify the user-selected policy: unreported effort is assigned to the maximum profile, while explicit source settings and measured scores are preserved. Counts distinguish direct fitted cells from displayed-only evidence. This is a declared assumption, not a claim that the underlying evaluator actually used its maximum setting.


In [1]:
import json, math, hashlib
from pathlib import Path
HERE=next(x for x in [Path.cwd(),Path.cwd()/"docs/audits/1.4.3-effort-coverage"] if (x/"audit-summary.json").exists())
def read(name):return json.loads((HERE/name).read_text())
s=read("audit-summary.json");p=read("accepted-input.json");d=read("accepted-diagnostics.json")
assert s["policy"]==p["unreported_effort_policy"]=="maximum"
assert hashlib.sha256((HERE/"accepted-input.json").read_bytes()).hexdigest()==s["accepted_input_sha256"]
print("Pinned input:",s["accepted_input_sha256"])


Pinned input: e5d654f41019df63f9f78c28efcd8d83db9bf425bf63e4d2cadb3a166fe23578


In [2]:
moved=read("moved-observations.json")
assert len(moved)==46
for change in moved:
 a,b=change["before"],change["after"]
 assert a["profile"]=="std-common" and b["profile"]=="max-common"
 assert b["effortAssumedMaximum"] and b["metadataIncomplete"]
 for key in ["score","standardError","effortTier","likelihood","variance","x","y","nTasks","nRuns"]:assert a[key]==b[key],key
 assert b["effortTier"] is None
print("46 existing observations moved; source values and likelihood measurements unchanged.")
assert s["newly_admitted_unreported_observations"]==3
print("Three previously unassigned observations admitted by the maximum assumption.")


46 existing observations moved; source values and likelihood measurements unchanged.
Three previously unassigned observations admitted by the maximum assumption.


In [3]:
f=read("focal-preparation.json")
maximum={r["benchmarkId"] for r in f if r["profile"]=="max-common"}
assert maximum=={"matharena-composite","deepswe-1.1","lmarena-text-style-controlled","terminal-bench-4.0","vals-finance-agent-v2-partial"}
assert next(r for r in f if r["benchmarkId"]=="matharena-composite")["effortAssumedMaximum"]
assert any(r["benchmarkId"]=="deepswe-1.1" and r["effortTier"]=="medium" and r["profile"]=="std-common" for r in f)
assert s["gemini_before"]["mixed"]["coverage"]==4 and s["gemini_after"]["mixed"]["coverage"]==5
assert p["n_benchmarks"]==19
assert s["models_with_greater_maximum_coverage"]==21
print("Gemini High coverage: 4/19 → 5/19. Twenty-one model releases gain direct maximum-profile coverage.")
print("Explicit Medium DeepSWE stays standard; source MathArena effort remains unreported.")


Gemini High coverage: 4/19 → 5/19. Twenty-one model releases gain direct maximum-profile coverage.
Explicit Medium DeepSWE stays standard; source MathArena effort remains unreported.


In [4]:
old=json.loads((HERE.parent/"1.4.2-gemini-coverage/accepted-input.json").read_text())
for k in s["unchanged_modeling_fields"]:assert old[k]==p[k],k
assert len(p["observations"])==924 and d["accepted"]
assert d["divergences"]==0 and d["posterior_draws"]>=12000
assert max(v["rhat"] for v in d["parameters"].values())<=1.01
assert min(min(v["ess_bulk"],v["ess_tail"]) for v in d["parameters"].values())>=400
assert min(d["ebfmi"])>=.3 and max(d["mcse_display_points"].values())<=.3
print("The existing formula, prior weights, calibration panel and benchmark conditions are unchanged.")
print(json.dumps(s["fit"],indent=2))


The existing formula, prior weights, calibration panel and benchmark conditions are unchanged.
{
  "observations": 924,
  "models": 110,
  "systems": 131,
  "benchmarks": 19,
  "retained_draws": 12000,
  "divergences": 0,
  "max_rhat": 1.0017593627588304,
  "min_ess": 2618.581751930913,
  "min_ebfmi": 0.8263949569647864,
  "max_mcse": 0.1970761863776125
}


In [5]:
for kind,row in s["gemini_after"].items():
 assert row["ci90"][0]<=row["median"]<=row["ci90"][1]
 print(kind,round(row["median"],2),"90% interval",row["ci90"],"coverage",row["coverage"])
print("All",s["source_rows_unchanged"],"source rows retain their measured values and source configurations.")


mixed 61.79 90% interval [56.458298, 67.800865] coverage 5
agentic 61.23 90% interval [55.572765, 68.277985] coverage 5
chat 60.93 90% interval [54.199615, 68.078445] coverage 5
All 1517 source rows retain their measured values and source configurations.


## Limits

Missing effort is now a maximum-profile assumption selected by the user. It can overstate the actual effort if an evaluator used an unreported lower setting. The original source field is not filled with a fabricated High label, and incomplete-configuration run noise remains active. Explicit levels, incompatible conditions, missing likelihood uncertainty and observed-only status retain their existing treatment. No new out-of-sample predictive superiority is claimed.
